# 模块2，第2节：基于评估驱动的发展

构建数据集并预先设置评估指标的好处在于，我们有一个定量信号，可以用来理解应用更改对其下游性能的影响。如果没有这个信号，我们将处于“飞行动作”的状态！

![评估流程图](../../images/eval_process.png)

### 系统性改进我们的应用程序

基于基准的评估揭示了当前数据库子代理的不足之处：

**问题：**
- **❌过多的工具调用**：连续的 `get_customer_orders` 调用 → 多次 `get_order_status` → 多次 `get_order_item_price` 等等。
- **💡缺乏能力**：无法聚合（SUM、COUNT、AVG）、过滤（WHERE）或跨表联结 - 因此依赖于内存计算
- **💡灵活性受限**：刚性工具无法适应复杂或新颖的查询

**可能的解决方案：**
构建一个灵活的SQL代理，使其能够在运行时自动生成定制查询以高效回答任何数据库问题。

让我们利用评估设置来推动有针对性的改进！

## 1. 初始化

我们将构建一个SQL代理，使其能够生成自定义查询以回答复杂的数据库问题。

<div align="center">
    <img src="../../images/db_agent_improvement.png">
</div>

In [ ]:
from dotenv import load_dotenv
from tools.database import get_database

load_dotenv()

# Get database connection (lazy loaded)
db = get_database()

# Extract schema once - we'll inject this into the agent's system prompt
table_info = db.get_table_info()

## 2. 建立SQL代理

使用`create_agent()`抽象方法创建一个能够即时生成所需SQL查询的代理工具。

In [ ]:
from langchain.tools import tool


@tool
def execute_sql(query: str) -> str:
    """Execute a SELECT query against the TechHub database.

    Safety: Only SELECT queries allowed - no INSERT/UPDATE/DELETE/etc.
    """
    # Safety check: Only allow SELECT queries
    if not query.strip().upper().startswith("SELECT"):
        return "Error: Only SELECT queries are allowed."

    # Block dangerous keywords
    FORBIDDEN = [
        "INSERT",
        "UPDATE",
        "DELETE",
        "ALTER",
        "DROP",
        "CREATE",
        "REPLACE",
        "TRUNCATE",
    ]
    if any(keyword in query.upper() for keyword in FORBIDDEN):
        return "Error: Query contains forbidden keyword."

    # Execute query
    db = get_database()  # lazy loaded
    try:
        result = db._execute(query)
        result = [tuple(row.values()) for row in result]  # extract values
        return result
    except Exception as e:
        return f"SQL Error: {str(e)}"

In [ ]:
SQL_AGENT_SYSTEM_PROMPT = f"""You are a database specialist for TechHub customer support.

You have access to a SQLite database with the following schema:

{table_info}

Your capabilities:
- Write SQL SELECT queries to answer any database question
- Use JOINs, aggregations (SUM, COUNT, AVG), filtering (WHERE), GROUP BY, ORDER BY
- Handle complex queries with multiple conditions

Guidelines:
1. Only use SELECT queries (read-only access)
2. Use proper JOINs when querying related tables
3. Format currency as $X.XX in your final answer
4. Provide context, not just raw numbers
5. If a query returns no results, explain why

Important: Read-only access - no INSERT/UPDATE/DELETE operations."""

In [ ]:
from langchain.agents import create_agent
from config import DEFAULT_MODEL

sql_agent = create_agent(
    model=DEFAULT_MODEL,
    tools=[execute_sql],
    name="sql_agent",
    system_prompt=SQL_AGENT_SYSTEM_PROMPT,
)
sql_agent

#### 生产安全注意事

在本次工作坊中，我们采用关键字过滤技术来防止写入操作。在生产环境下的PostgreSQL数据库中，应实施多层防御：

1. **数据库级**：创建一个只读用户
2. **连接级**：使用SQLAlchemy的`postgresql_readonly=True`参数（例如）
3. **应用级**：在执行前验证查询（目前我们的方法）

SQLite不支持用户角色，因此我们采用关键字过滤的方法在本次演示中进行教学和功能实现。

## 3. 简单演示

让我们用 SQL 智能代理来测试之前在基准评估中失败的查询。

In [ ]:
# Test on a query that requires aggregation
question = "What items were in order ORD-2023-0002? How much did each cost?"

result = sql_agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
)

for message in result["messages"]:
    message.pretty_print()

✅ **成功！** SQL代理生成带有JOIN和聚合函数的自定义查询，能够在单次工具调用中高效地回答问题。

查看LangSmith轨迹以查看生成的SQL查询！

## 4. 在全 supervisor HITL 系统中部署 SQL 节点

现在，我们将我们旧的僵硬的数据库代理换成了灵活的 SQL 代理，在完整的 Supervisor HITL 系统中。

In [ ]:
from agents.docs_agent import create_docs_agent
from agents.sql_agent import create_sql_agent
from agents.supervisor_hitl_agent import create_supervisor_hitl_agent

# Instantiate improved SQL agent for deployment
sql_agent = create_sql_agent()

# Instantiate docs agent
docs_agent = create_docs_agent()


# Compose supervisor HITL with SQL agent instead of old db_agent
improved_agent = create_supervisor_hitl_agent(
    db_agent=sql_agent,
    docs_agent=docs_agent,
)

In [ ]:
from IPython.display import Image

display(Image(improved_agent.get_graph(xray=True).draw_mermaid_png()))

快速测试：

In [ ]:
import uuid
from langgraph.types import Command

# New thread
thread_id = uuid.uuid4()
config = {"configurable": {"thread_id": thread_id}}

# First invocation - will pause at interrupt
result = improved_agent.invoke(
    {"messages": [{"role": "user", "content": "Whats the status of my recent order?"}]},
    config=config,
)

result

In [ ]:
# Resume with valid email
result = improved_agent.invoke(
    Command(resume="Ok, its: sarah.chen@gmail.com"),
    config=config,
)
result

In [ ]:
for message in result["messages"]:
    message.pretty_print()

## 5. 重新评估绩效

现在，让我们在同一个 dataset 上运行相同的 Evaluation，以定量地了解它如何影响我们的性能指标。

In [ ]:
# Import evaluators from Section 1 (now packaged in evaluators module)
from evaluators import correctness_evaluator, count_total_tool_calls_evaluator

In [ ]:
import uuid
from langsmith import Client

client = Client()


def improved_target_function(inputs: dict) -> dict:
    """Target function that wraps our improved agent for evaluation."""
    thread_id = uuid.uuid4()
    config = {"configurable": {"thread_id": thread_id}}

    result = improved_agent.invoke(
        inputs,
        config=config,
    )
    return {
        "messages": [{"role": "assistant", "content": result["messages"][-1].content}]
    }


# Get the most recent dataset name
most_recent_dataset_name = max(
    client.list_datasets(), key=lambda ds: ds.created_at
).name

# Run evaluation on same dataset
results = client.evaluate(
    improved_target_function,
    data=most_recent_dataset_name,  # Same dataset as section 1
    evaluators=[correctness_evaluator, count_total_tool_calls_evaluator],
    experiment_prefix="with-sql-agent-eval",
    description="Evaluate SQL agent's flexible query generation vs baseline rigid tools",
    max_concurrency=5,
)

## 6. 比较 LangSmith 中的结果

现在让我们使用 LangSmith 的比较功能来分析我们的改进。